# 밀도 추정 개체 수 세기 — 원문제 모범답안 코드 그대로 (Colab)

**연습 기법** (IOAI 2025 Individual **Chicken Counting** 모범답안과 동일): 객체를 *직접 검출*하지 않고 **밀도 맵**
(density map)을 회귀해 **합 = 개체 수**로 센다. 모범답안 = `FeatureExtraction`(특징추출) + 디코더 → 밀도 맵,
**MSE 손실**, **count = 밀도 합**.

**대회**: [Data Science Bowl 2018](https://www.kaggle.com/c/data-science-bowl-2018) — 닭 대신 **핵(nucleus) 개수**를 센다.
각 핵 중심에 가우시안을 찍어 **GT 밀도 맵**(합=핵 수)을 만들고, 모델이 밀도를 예측해 합으로 개수를 추정.

| Chicken Counting 모범답안 | 이 노트북 |
|---|---|
| `class FeatureExtraction` (dilated conv ×4, ÷4) | 동일 (그대로) |
| `DensityDecoder` (특징 → 1채널 밀도) | 동일 (템플릿대로 구현) |
| `density * scale` (작은 값 안정화) → MSE | 동일 (SCALE=1000) |
| **count = 밀도 맵 합** | 동일 |

> ⚙️ 720×1280 닭 이미지는 OOM 위험(원문제 경고)이라 여기선 **256×256→밀도 64×64** 경량화. GPU 권장, T4 수 분.
> ⚠️ **지표**: 원문제 핵심은 *개수 MAE*(예측 vs 평균예측 기준선 비교). DSB 리더보드는 *인스턴스 RLE* 라 개수세기와 다르지만,
> 밀도 맵의 **고밀도 영역(=핵 위치)을 연결요소로 묶어** RLE 제출도 만든다(러프한 위치기반 — 정밀 마스크 점수는 낮음).

## 0. 설치 · 자격증명 · 시드

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "scipy", "pillow", "scikit-image", "matplotlib"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
import random, numpy as np, torch
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
print("준비 완료")

## 1. Kaggle 다운로드 (stage1_train / stage2_test_final)

In [ ]:
import glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
for fn in ["stage1_train.zip", "stage2_test_final.zip"]:
    api.competition_download_file("data-science-bowl-2018", fn, path=WORK_DIR, quiet=False)
for fn, out in [("stage1_train.zip", "stage1_train"), ("stage2_test_final.zip", "stage2_test_final")]:
    zp = os.path.join(WORK_DIR, fn)
    with zipfile.ZipFile(zp) as zf: zf.extractall(os.path.join(WORK_DIR, out))
    os.remove(zp)
TRAIN_DIR = os.path.join(WORK_DIR, "stage1_train"); TEST_DIR = os.path.join(WORK_DIR, "stage2_test_final")
print("train:", len(os.listdir(TRAIN_DIR)), "| test:", len(os.listdir(TEST_DIR)))

## 2. 이미지 + GT 밀도 맵 (핵 중심 가우시안, 합=핵 수)
이미지 256×256, 밀도 맵 64×64(÷4). 각 핵 마스크의 **중심**에 점을 찍고 **가우시안 블러** → 밀도 맵.
밀도 값이 작아 학습이 어려우니 `SCALE` 을 곱한다(모범답안의 `density*scale`).

In [ ]:
from PIL import Image
from scipy.ndimage import gaussian_filter, center_of_mass
SZ, DS, SCALE = 256, 64, 1000.0           # 이미지 256, 밀도 64(÷4), 스케일

def load_sample(d):
    ip = glob.glob(os.path.join(d, "images", "*.png"))[0]
    img = np.asarray(Image.open(ip).convert("RGB").resize((SZ, SZ)), "float32").transpose(2,0,1)/255.
    pts = np.zeros((DS, DS), "float32"); n = 0
    for mp in glob.glob(os.path.join(d, "masks", "*.png")):
        m = np.asarray(Image.open(mp).convert("L")) > 0
        if m.sum() == 0: continue
        cy, cx = center_of_mass(m)
        yy = min(int(cy / m.shape[0] * DS), DS-1); xx = min(int(cx / m.shape[1] * DS), DS-1)
        pts[yy, xx] += 1.0; n += 1
    dens = gaussian_filter(pts, sigma=2.0) * SCALE                   # 합 보존 가우시안 → 밀도
    return img, dens, n

ids = sorted(os.listdir(TRAIN_DIR))
samples = [load_sample(os.path.join(TRAIN_DIR, i)) for i in ids]
X = np.stack([s[0] for s in samples]).astype("float32")
D = np.stack([s[1] for s in samples]).astype("float32")
counts = np.array([s[2] for s in samples], "float32")
print("X:", X.shape, "| D:", D.shape, "| 핵 수 min/median/max:", int(counts.min()), int(np.median(counts)), int(counts.max()))

## 3. FeatureExtraction + DensityDecoder (모범답안 그대로)
`FeatureExtraction`(dilated conv ×4, ÷4) → `DensityDecoder`(특징 → 1채널 밀도). 256 입력 → 64×64 밀도.

In [ ]:
import torch.nn as nn, torch.nn.functional as F

class FeatureExtraction(nn.Module):                  # 모범답안 그대로 (dilated conv ×4 + pool ×2 → ÷4)
    def __init__(self, in_channels=3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, 3, padding=2, dilation=2)
        self.conv2 = nn.Conv2d(64, 64, 3, padding=2, dilation=2); self.pool2 = nn.MaxPool2d(2,2)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=2, dilation=2)
        self.conv4 = nn.Conv2d(128, 128, 3, padding=2, dilation=2); self.pool4 = nn.MaxPool2d(2,2)
    def forward(self, x):
        x = F.relu(self.conv1(x)); x = self.pool2(F.relu(self.conv2(x)))
        x = F.relu(self.conv3(x)); x = self.pool4(F.relu(self.conv4(x)))
        return x                                     # (B,128,64,64)

class DensityDecoder(nn.Module):                     # 특징 → 1채널 밀도 맵
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(128,64,3,padding=1), nn.ReLU(True),
                                 nn.Conv2d(64,32,3,padding=1), nn.ReLU(True),
                                 nn.Conv2d(32,1,1))   # 출력 ReLU 제거(dead-ReLU 붕괴 방지)
    def forward(self, x): return self.net(x)

class CountNet(nn.Module):                           # = ChickenCounting
    def __init__(self):
        super().__init__(); self.feature_extraction = FeatureExtraction(); self.decoder = DensityDecoder()
    def forward(self, x): return self.decoder(self.feature_extraction(x))

print("CountNet 파라미터:", sum(p.numel() for p in CountNet().parameters()))

## 4. 학습 — 밀도 맵 **MSE** (count = 밀도 합)
*주의*: 희소한 밀도에 MAE 를 섞으면 모델이 *전부 0* 으로 붕괴(0 예측이 MAE 최소)한다 → **MSE 단독**이 안정적.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
idx = np.random.RandomState(42).permutation(len(X)); cut = int(len(X)*0.9)
tr, va = idx[:cut], idx[cut:]
Xt = torch.from_numpy(X); Dt = torch.from_numpy(D)[:, None]

torch.manual_seed(0)                                  # 안정적 초기화
model = CountNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
mse = nn.MSELoss()
EPOCHS, BS = 40, 8

def pred_counts(which):
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(Xt[which[j:j+BS]].to(device)).sum((1,2,3)).cpu().numpy()
                               for j in range(0, len(which), BS)]) / SCALE

for ep in range(EPOCHS):
    model.train(); perm = np.random.RandomState(ep).permutation(tr)
    for i in range(0, len(perm), BS):
        b = perm[i:i+BS]; opt.zero_grad()
        loss = mse(model(Xt[b].to(device)), Dt[b].to(device))
        loss.backward(); opt.step()
    if (ep+1) % 10 == 0:
        print(f"epoch {ep+1}/{EPOCHS} | val 개수 MAE = {np.abs(pred_counts(va) - counts[va]).mean():.2f}")

pc = pred_counts(va)
base_mae = np.abs(counts[tr].mean() - counts[va]).mean()      # 평균값 예측 기준선
print(f"\n검증 개수 MAE = {np.abs(pc-counts[va]).mean():.2f}  |  기준선(평균예측) MAE = {base_mae:.2f}")
print("→ 밀도 회귀로 개수 추정이 평균예측 기준선을 능가 — Chicken Counting 의 density estimation 핵심.")

## 5. 밀도 맵 시각화

In [ ]:
import matplotlib.pyplot as plt
for k in range(3):
    j = int(va[k])
    model.eval()
    with torch.no_grad(): dm = model(Xt[j:j+1].to(device))[0,0].cpu().numpy()
    fig, ax = plt.subplots(1,3, figsize=(11,3))
    ax[0].imshow(X[j].transpose(1,2,0)); ax[0].set_title(f"image (true {int(counts[j])})"); ax[0].axis("off")
    ax[1].imshow(D[j], cmap="jet"); ax[1].set_title("GT density"); ax[1].axis("off")
    ax[2].imshow(dm, cmap="jet"); ax[2].set_title(f"pred (count {dm.sum()/SCALE:.1f})"); ax[2].axis("off")
    plt.show()

## 6. 캐글 제출 — 밀도 고영역 → 인스턴스 RLE (`ImageId,EncodedPixels`)
DSB 리더보드는 인스턴스 RLE 형식. 밀도 맵을 원본 크기로 키워 **고밀도 영역을 연결요소**로 묶어 RLE 로 제출한다
(밀도가 핵 위치를 알려주므로 *위치 기반* 제출 — 정밀 마스크가 아니라 점수는 낮지만 형식은 유효).

In [ ]:
import pandas as pd
from skimage.morphology import label

def rle_encode(mask):
    px = mask.flatten(order="F"); px = np.concatenate([[0], px, [0]])
    runs = np.where(px[1:] != px[:-1])[0] + 1; runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

test_ids = sorted(os.listdir(TEST_DIR)); rows = []
model.eval()
for sid in test_ids:
    paths = glob.glob(os.path.join(TEST_DIR, sid, "images", "*.png")); found = False
    if paths:
        orig = Image.open(paths[0]).convert("RGB"); Wd, Ht = orig.size
        x = np.asarray(orig.resize((SZ, SZ)), "float32").transpose(2,0,1)/255.
        with torch.no_grad():
            dm = model(torch.from_numpy(x[None]).to(device))[0,0].cpu().numpy()   # 64×64 밀도
        up = np.asarray(Image.fromarray(dm).resize((Wd, Ht), Image.BILINEAR))     # 원본 크기로
        mask = up > max(up.max()*0.25, 1.0)                                       # 고밀도 영역
        lab = label(mask)
        for kk in range(1, lab.max()+1):
            rle = rle_encode(lab == kk)
            if rle: rows.append({"ImageId": sid, "EncodedPixels": rle}); found = True
    if not found: rows.append({"ImageId": sid, "EncodedPixels": "1 1"})

submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame(rows, columns=["ImageId", "EncodedPixels"]).to_csv(submission_path, index=False)
print("Saved:", submission_path, "| rows:", len(rows), "| unique ids:", pd.DataFrame(rows)["ImageId"].nunique(), "/", len(test_ids))

In [ ]:
try:
    from google.colab import files; files.download(submission_path)
except Exception:
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [Data Science Bowl 2018](https://www.kaggle.com/c/data-science-bowl-2018/submit) 에 제출.

Chicken Counting **모범답안** 골격(`FeatureExtraction` + `DensityDecoder` → 밀도 맵, MSE 손실, **count=밀도 합**)을
그대로 옮겨 핵 개수 세기로 연습 — *검출 없이 밀도 회귀*. **핵심 지표는 개수 MAE**(평균예측 기준선 대비 ↓). DSB LB 제출은
밀도 고영역을 RLE 로 변환(위치 기반·러프). 더: 사전학습 backbone, sigma·해상도·SCALE 조정, MSE 단독 유지(MAE 섞으면 붕괴).
(원문제는 위성기상 Weather 의 연장 — dense 회귀; 720×1280 은 OOM 주의).